# 13 — Accuracy Validation

**Objective**: consolidate Section 13's ten accuracy checks into one inspectable report (`src/services/accuracy_audit.py`), run it against a real pipeline result, and prove each check actually catches the condition it exists to catch — not just that it passes on well-formed input.

**Dependencies**: everything through Phase 10. Every one of the ten checks was already enforced somewhere in the pipeline before this phase; this notebook is about consolidation and independent re-verification, not new rules.

**Two real bugs this consolidation caught, not merely aspirational gaps:**
1. `freshness_thresholds_hours.memory` existed in `config/risk_rules.yaml` since Phase 1 — nothing ever read it. Fixed with `reconciliation.detect_stale_memory`, now live in `validate_findings`.
2. `reconciliation.detect_reconciliation_failure` stamped `Risk.project_id` from the raw finance-system id (`"10001"`) instead of the `project_mapping.yaml` key (`"PROJECT-10001"`) — the exact same bug already fixed in `financial_metrics.py` back in Phase 8, missed here. Every reconciliation-failure finding from a real run was invisible in any per-project view (Financial Health page, chat answers) despite still correctly degrading portfolio-wide confidence. Caught live the moment `validate_findings`' new project-reference check ran against a real graph result — not found by reading code, found by running it.

In [1]:
import os
import sys
import shutil
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

shutil.rmtree(PROJECT_ROOT / "data/snapshots", ignore_errors=True)
(PROJECT_ROOT / "data/snapshots").mkdir(parents=True, exist_ok=True)

from datetime import datetime, timezone

from src.connectors.jira_client import build_default_jira_client
from src.connectors.financial_client import CSVFinancialDataSource
from src.services import project_unifier, accuracy_audit
from src.services.memory_store import FileMemoryStore
from src.graph.nodes import NodeDeps
from src.graph.workflow import build_graph

deps = NodeDeps(
    jira_client=build_default_jira_client(),
    financial_source=CSVFinancialDataSource(),
    memory_store=FileMemoryStore(),
    mapping=project_unifier.load_project_mapping(),
)
graph = build_graph(deps)
result = graph.invoke({
    "user_question": "What is the status of our portfolio?",
    "request_id": "req-001",
    "requested_at": datetime(2026, 9, 15, tzinfo=timezone.utc),
})
print("Pipeline run complete.")

{"request_id": "req-001", "timestamp": "2026-09-03T03:23:09.065659+00:00", "event": "graph_node_start", "node": "classify_request"}
{"request_id": "req-001", "timestamp": "2026-09-03T03:23:09.066083+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.02}
{"request_id": "req-001", "timestamp": "2026-09-03T03:23:09.066624+00:00", "event": "graph_node_start", "node": "fetch_delivery_data"}
{"request_id": "req-001", "timestamp": "2026-09-03T03:23:09.084692+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 270, "sprint_count": 20, "partial_failure": false}
{"request_id": "req-001", "timestamp": "2026-09-03T03:23:09.084765+00:00", "event": "graph_node_end", "node": "fetch_delivery_data", "latency_ms": 18.1}
{"request_id": "req-001", "timestamp": "2026-09-03T03:23:09.085318+00:00", "event": "graph_node_start", "node": "validate_delivery_data"}
{"request_id": "req-001", "timestamp": "2026-09-03T03:23:09.085415+00:00", "event": "graph_node_end", "

## The consolidated audit, against a real run

In [2]:
report = accuracy_audit.run_accuracy_audit(result)
print(report.summary())
assert report.all_passed

Accuracy Audit — ALL CHECKS PASSED
  [PASS] 1. Source Provenance: all 7 project(s) carry provenance consistent with their mapping status
  [PASS] 2. Freshness: tracked: Jira, Finance, Memory
  [PASS] 3. Data Completeness: 2 unmapped project(s), all surfaced
  [PASS] 4. Financial Reconciliation: 6 financial record(s) all internally consistent
  [PASS] 5. Deterministic Calculations: 7 project(s) checked, all consistent
  [PASS] 6. Historical Verification: 7 project(s) checked, no unsupported historical claims
  [PASS] 7. Evidence Requirement: every HIGH-severity risk carries evidence
  [PASS] 8. Confidence: confidence='LOW CONFIDENCE'
  [PASS] 9. No Hallucination: every risk and citation traces to a real source
  [PASS] 10. Validation Node: validation_passed=True, 3 check(s) recorded


## Reproducing the bug this phase found

Before the fix, `detect_reconciliation_failure` used `record.project_id` (e.g. `"10001"`) unconditionally. Here's what that looked like, and why Check 9 would have caught it immediately had it existed at the time.

In [3]:
from src.connectors.financial_client import CSVFinancialDataSource
from src.services import reconciliation

fin_source = CSVFinancialDataSource()
record = fin_source.get_project_finances("10001", "2026-01").with_calculated_fields()

before_fix = reconciliation.detect_reconciliation_failure(record)  # no project_id override -> old buggy behavior
after_fix = reconciliation.detect_reconciliation_failure(record, project_id="PROJECT-10001")  # what the graph actually passes now

print(f"Without override (the old bug's shape): project_id={before_fix.project_id!r}")
print(f"With override (what the graph passes):  project_id={after_fix.project_id!r}")
print()
print("The 'before' value would never match any entry in unified_projects (which uses the mapping key),")
print("so it would render as an orphan under Check 9 and be invisible in every per-project view.")

Without override (the old bug's shape): project_id='10001'
With override (what the graph passes):  project_id='PROJECT-10001'

The 'before' value would never match any entry in unified_projects (which uses the mapping key),
so it would render as an orphan under Check 9 and be invisible in every per-project view.


## Proving each check actually catches its failure mode

A check that never fails on anything is not verified — it's just unexercised. Each cell below deliberately breaks one invariant and confirms the matching check reports it.

In [4]:
from src.models.risk import Risk
from src.models.common import RiskSeverity

# Check 7: HIGH severity with no evidence
broken = dict(result)
broken["risks"] = result["risks"] + [
    Risk(risk_id="FAKE-NO-EVIDENCE", project_id="PORTFOLIO", category="Delivery", severity=RiskSeverity.HIGH, description="fabricated", evidence=[])
]
check7 = accuracy_audit.check_7_evidence_requirement(broken)
print(f"Check 7 (evidence requirement): passed={check7.passed}")
print(f"  {check7.detail}")
assert not check7.passed

Check 7 (evidence requirement): passed=False
  1 HIGH-severity risk(s) with no evidence: ['FAKE-NO-EVIDENCE']


In [5]:
# Check 9: a risk referencing a project that doesn't exist
broken = dict(result)
broken["risks"] = result["risks"] + [
    Risk(risk_id="FAKE-ORPHAN", project_id="PROJECT-DOES-NOT-EXIST", category="Delivery", severity=RiskSeverity.LOW, description="fabricated", evidence=["e"])
]
check9 = accuracy_audit.check_9_no_hallucination(broken)
print(f"Check 9 (no hallucination): passed={check9.passed}")
print(f"  {check9.detail}")
assert not check9.passed

Check 9 (no hallucination): passed=False
  orphan risks: ['FAKE-ORPHAN']; malformed citations: 0


In [6]:
# Check 4: a remaining_budget that doesn't match approved - actual - committed
tampered = result["financial_data"][0].model_copy(update={"remaining_budget": 999_999_999.0})
broken = dict(result)
broken["financial_data"] = [tampered] + result["financial_data"][1:]
check4 = accuracy_audit.check_4_financial_reconciliation(broken)
print(f"Check 4 (financial reconciliation): passed={check4.passed}")
print(f"  {check4.detail}")
assert not check4.passed

Check 4 (financial reconciliation): passed=False
  10001: remaining_budget does not match approved-actual-committed


In [7]:
# Check 2: freshness tracking missing entirely
broken = dict(result)
broken["jira_data_freshness"] = None
check2 = accuracy_audit.check_2_freshness(broken)
print(f"Check 2 (freshness): passed={check2.passed}")
print(f"  {check2.detail}")
assert not check2.passed

Check 2 (freshness): passed=False
  tracked: Finance, Memory; missing: Jira


## Memory freshness, live

The other Phase 11 gap: `freshness_thresholds_hours.memory` is finally read. A second, much later run shows it firing on real accumulated history.

In [8]:
later_result = graph.invoke({
    "user_question": "What is the status of our portfolio?",
    "request_id": "req-002",
    "requested_at": datetime(2027, 6, 1, tzinfo=timezone.utc),
})
memory_stale_findings = [r for r in later_result["risks"] if "MEMORY_STALE" in r.risk_id]
print(f"{len(memory_stale_findings)} memory-staleness finding(s) on the later run")
for r in memory_stale_findings[:2]:
    print(f"  {r.description}")

later_report = accuracy_audit.run_accuracy_audit(later_result)
print()
print(later_report.summary())
assert later_report.all_passed  # still passes — memory staleness degrades confidence, it isn't a structural failure

{"request_id": "req-002", "timestamp": "2026-09-03T03:23:09.185981+00:00", "event": "graph_node_start", "node": "classify_request"}
{"request_id": "req-002", "timestamp": "2026-09-03T03:23:09.186148+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.01}
{"request_id": "req-002", "timestamp": "2026-09-03T03:23:09.186539+00:00", "event": "graph_node_start", "node": "fetch_delivery_data"}
{"request_id": "req-002", "timestamp": "2026-09-03T03:23:09.204450+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 270, "sprint_count": 20, "partial_failure": false}
{"request_id": "req-002", "timestamp": "2026-09-03T03:23:09.204522+00:00", "event": "graph_node_end", "node": "fetch_delivery_data", "latency_ms": 17.94}
{"request_id": "req-002", "timestamp": "2026-09-03T03:23:09.205005+00:00", "event": "graph_node_start", "node": "validate_delivery_data"}
{"request_id": "req-002", "timestamp": "2026-09-03T03:23:09.205092+00:00", "event": "graph_node_end", 

{"request_id": "req-002", "timestamp": "2026-09-03T03:23:09.282263+00:00", "event": "graph_node_end", "node": "validate_findings", "latency_ms": 33.34}
{"request_id": "req-002", "timestamp": "2026-09-03T03:23:09.282829+00:00", "event": "graph_node_start", "node": "generate_response"}


{"request_id": "req-002", "timestamp": "2026-09-03T03:23:09.350092+00:00", "event": "graph_node_end", "node": "generate_response", "latency_ms": 67.22}
{"request_id": "req-002", "timestamp": "2026-09-03T03:23:09.350986+00:00", "event": "graph_node_start", "node": "persist_snapshot"}
{"request_id": "req-002", "timestamp": "2026-09-03T03:23:09.353410+00:00", "event": "datasource_access", "source": "Memory", "operation": "persist_snapshot", "written_count": 7}
{"request_id": "req-002", "timestamp": "2026-09-03T03:23:09.353466+00:00", "event": "graph_node_end", "node": "persist_snapshot", "latency_ms": 2.4}
7 memory-staleness finding(s) on the later run
  Most recent stored snapshot for this project (2026-09-15) is 259 days old
  Most recent stored snapshot for this project (2026-09-15) is 259 days old

Accuracy Audit — ALL CHECKS PASSED
  [PASS] 1. Source Provenance: all 7 project(s) carry provenance consistent with their mapping status
  [PASS] 2. Freshness: tracked: Jira, Finance, Memor

## Validation checks

- [x] All ten checks pass against a real, unmodified graph result
- [x] Each check individually catches its own failure mode when the corresponding invariant is deliberately broken — not just verified to pass
- [x] The reconciliation project_id bug is reproduced and shown fixed, with the exact mechanism explained
- [x] Memory freshness — configured since Phase 1, unused until now — correctly fires as a confidence-degrading (not structural) finding

## Testing

`tests/test_accuracy_audit.py` (15 tests) — one full-pipeline pass/fail regression plus one injected-failure test per check.

## Next step

Phase 12: end-to-end testing and production hardening — `14_end_to_end_testing.ipynb`.